In [0]:
%run "/Workspace/Repos/shoyofromconcrete@gmail.com/MultiChannelDataPipeLine/config"


In [0]:
from pyspark.sql.functions import*
from pyspark.sql.window import*
from pyspark.sql.types import*

In [0]:
silver_df=spark.table("multidatadumps.silver.sales_silver")
display(silver_df)

In [0]:
calculations_df = (
    silver_df

    # Fix nulls FIRST
    .withColumn("discount", coalesce(col("discount"), lit(0)))
    .withColumn("tax", coalesce(col("tax"), lit(0)))
    .withColumn("quantity", coalesce(col("quantity"), lit(0)))

    # Calculations
    .withColumn("final_amount", col("total_amount") - col("discount") - col("tax"))
    .withColumn("gross_amount", col("price_per_unit") * col("quantity"))
    .withColumn("gross_final_diff", col("gross_amount") - col("final_amount"))

    # Date features
    .withColumn("order_month", month(col("order_date")))
    .withColumn("order_year", year(col("order_date")))

    # Category
    .withColumn(
        "order_value_category",
        when(col("final_amount") < 1000, "Low")
        .when(col("final_amount").between(1000, 5000), "Medium")
        .otherwise("High")
    )

    # Valid order flag
    .withColumn(
        "is_valid_order",
        when(
            (col("order_status") != "Cancelled") &
            (col("payment_status") != "Failed") &
            (col("final_amount") > 0) &
            (col("quantity") > 0),
            "Valid"
        ).otherwise("Invalid")
    )
)

calculations_df.display()
calculations_df.printSchema()


In [0]:
def fill_nulls_by_type(df):

    fill_dict = {}

    for col_name, dtype in df.dtypes:

        # Numeric types
        if dtype in ["int", "bigint", "double", "float", "decimal"]:
            fill_dict[col_name] = 0

        # String type
        elif dtype == "string":
            fill_dict[col_name] = "unknown"

    return df.fillna(fill_dict)

In [0]:
calculations_df_clean=fill_nulls_by_type(calculations_df)

In [0]:
final_schema_list=[
    "order_id",
    "customer_name",
    "email",
    "phone",
    "product_name",
    "category",
    "quantity",
    "price_per_unit",
    "gross_amount",
    "discount",
    "tax",
    "final_amount",
    "payment_method",
    "payment_status",
    "order_status",
    "order_date",
    "delivery_date",
    "order_month",
    "order_year",
    "shipping_city",
    "pincode",
    "source",
    "order_value_category",
    "is_valid_order",
    "source_file"
]

In [0]:
fact_sales=calculations_df_clean.select(final_schema_list)
fact_sales=fact_sales.filter(col("is_valid_order")=="Valid")
display(fact_sales)
